In [31]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path().resolve().parent
sys.path.append(str(project_root))
 
from src.graph import build_graph, basic_stats, build_graphs_by_book, build_cumulative_graph
from src.features import extract_features, extract_temporal_features, aggregate_temporal_features, add_trend_features
from src.data_loader import load_edges, load_all_edges, load_node_labels

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [32]:
df = load_all_edges()

In [33]:
df.head()

,Source,Target,Type,weight,book
0,Addam-Marbrand,Jaime-Lannister,Undirected,3,1
1,Addam-Marbrand,Tywin-Lannister,Undirected,6,1
2,Aegon-I-Targaryen,Daenerys-Targaryen,Undirected,5,1
3,Aegon-I-Targaryen,Eddard-Stark,Undirected,4,1
4,Aemon-Targaryen-(Maester-Aemon),Alliser-Thorne,Undirected,4,1


In [34]:
G = build_graphs_by_book(df)

print(G)

{1: <networkx.classes.graph.Graph object at 0x000002259F564BC8>, 2: <networkx.classes.graph.Graph object at 0x000002259F5609C8>, 3: <networkx.classes.graph.Graph object at 0x000002259F41BDC8>, 4: <networkx.classes.graph.Graph object at 0x000002259F470348>, 5: <networkx.classes.graph.Graph object at 0x000002259C206B88>}


In [35]:
print(df["book"].value_counts())

3    1008
2     775
5     760
1     684
4     682
Name: book, dtype: int64


In [36]:
df = df.drop(columns = ['Type'], axis = 0)

In [37]:
print(df.columns)

Index(['Source', 'Target', 'weight', 'book'], dtype='object')


In [38]:
graphs = build_graphs_by_book(df)

In [39]:
df_temporal = extract_temporal_features(graphs)

In [40]:
df_temporal.head()

,node,book,degree,betweenness,pagerank
0,Addam-Marbrand,1,0.010753,0.00000,0.001276
1,Jaime-Lannister,1,0.155914,0.03201,0.014403
2,Tywin-Lannister,1,0.118280,0.02619,0.011424
3,Aegon-I-Targaryen,1,0.010753,0.00000,0.001254
4,Daenerys-Targaryen,1,0.112903,0.08627,0.027099


In [41]:
df_agg = aggregate_temporal_features(df_temporal)
df_trend = add_trend_features(df_temporal)

In [42]:
final_features = df_agg.merge(df_trend, on="node")
labels = load_node_labels()

In [43]:
labels.head()
labels.shape

(1557, 2)

In [44]:
final_features["node"] = final_features["node"].astype(str).str.strip()
labels["name"] = labels["name"].astype(str).str.strip()
labels.head()

,name,isAlive
0,Viserys II Targaryen,0
1,Walder Frey,1
2,Addison Hill,1
3,Aemma Arryn,0
4,Sylva Santagar,1


In [45]:
dataset = final_features.merge(
    labels,
    left_on="node",
    right_on="name",
    how="inner"
)

In [46]:
print(dataset.shape)
dataset.head()

(181, 11)


,node,degree_mean,degree_max,betweenness_mean,betweenness_max,pagerank_mean,pagerank_max,degree_change,pagerank_change,name,isAlive
0,Aggar,0.015504,0.015504,0.024813,0.024813,0.003762,0.003762,0.000000,0.000000,Aggar,0
1,Aggo,0.019312,0.032258,0.000031,0.000073,0.002785,0.004259,-0.013271,-0.000759,Aggo,1
2,Albett,0.016129,0.016129,0.000000,0.000000,0.001484,0.001484,0.000000,0.000000,Albett,1
3,Alebelly,0.015504,0.015504,0.000040,0.000040,0.001875,0.001875,0.000000,0.000000,Alebelly,0
4,Amabel,0.007752,0.007752,0.000000,0.000000,0.001550,0.001550,0.000000,0.000000,Amabel,1


In [47]:
print(dataset["isAlive"].value_counts())
print(labels["isAlive"].value_counts())

1    119
0     62
Name: isAlive, dtype: int64
1    1212
0     345
Name: isAlive, dtype: int64


In [48]:
print("final_features:", final_features.shape)
print("labels:", labels.shape)
print("dataset after merge:", dataset.shape)

final_features: (796, 9)
labels: (1557, 2)
dataset after merge: (181, 11)


In [49]:
X = dataset.drop(columns=["node", "name", "Death", "isAlive"], errors="ignore")
y = dataset["isAlive"]

In [50]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [51]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight='balanced',
                       criterion='gini', max_depth=5, max_features='auto',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=200,
                       n_jobs=None, oob_score=False, random_state=42, verbose=0,
                       warm_start=False)

In [52]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.39      0.54      0.45        13
           1       0.68      0.54      0.60        24

    accuracy                           0.54        37
   macro avg       0.54      0.54      0.53        37
weighted avg       0.58      0.54      0.55        37

[[ 7  6]
 [11 13]]


In [53]:
feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance

pagerank_max        0.208200
pagerank_mean       0.192602
degree_mean         0.170688
degree_max          0.169029
betweenness_mean    0.086706
betweenness_max     0.081308
pagerank_change     0.045808
degree_change       0.045660
dtype: float64

In [54]:
final_features["node"] = final_features["node"].astype(str).str.strip()
labels["name"] = labels["name"].astype(str).str.strip()

dataset = final_features.merge(labels, left_on="node", right_on="name", how="inner")

print(dataset.shape)
print(dataset["isAlive"].value_counts())

X = dataset.drop(columns=["node", "name", "Death", "isAlive"], errors="ignore")
y = dataset["isAlive"]

(181, 11)
1    119
0     62
Name: isAlive, dtype: int64


In [55]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.39      0.54      0.45        13
           1       0.68      0.54      0.60        24

    accuracy                           0.54        37
   macro avg       0.54      0.54      0.53        37
weighted avg       0.58      0.54      0.55        37



In [56]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

print(classification_report(y_test, dummy_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        13
           1       0.65      1.00      0.79        24

    accuracy                           0.65        37
   macro avg       0.32      0.50      0.39        37
weighted avg       0.42      0.65      0.51        37



E:\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1272: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [57]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42,
    class_weight="balanced"
)

scores = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=["accuracy", "f1", "precision", "recall"]
)

for metric, values in scores.items():
    if metric.startswith("test_"):
        print(metric, values.mean(), values.std())

test_accuracy 0.6081081081081081 0.043737272263151654
test_f1 0.6827635022244859 0.040262741215558395
test_precision 0.7326333783175889 0.050355068885577274
test_recall 0.6467391304347826 0.07859897443204035


In [58]:
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=2000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    logreg,
    X,
    y,
    cv=cv,
    scoring=["accuracy", "f1", "precision", "recall"]
)

for metric, values in scores.items():
    if metric.startswith("test_"):
        print(metric, values.mean(), values.std())

test_accuracy 0.5304804804804805 0.04845827132611907
test_f1 0.5962839093273876 0.06133040369208207
test_precision 0.6916666666666667 0.07987306274604367
test_recall 0.5369565217391303 0.09595363124382675


In [59]:
label_names = set(labels["name"].astype(str).str.strip())
feature_names = set(final_features["node"].astype(str).str.strip())

print("Names in labels:", len(label_names))
print("Names in final_features:", len(feature_names))
print("Intersection:", len(label_names & feature_names))
print("Only in labels:", len(label_names - feature_names))
print("Only in final_features:", len(feature_names - label_names))

Names in labels: 1557
Names in final_features: 796
Intersection: 181
Only in labels: 1376
Only in final_features: 615


In [61]:
G_4 = build_cumulative_graph(df, max_book=4)
features_4 = extract_features(G_4)

print(features_4.shape)
features_4.head()

AttributeError: 'dict' object has no attribute 'shape'